In [1]:
import uproot
import awkward as ak
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import math
import hist
import vector
import os
import subprocess
import gc
print("uproot version",uproot.__version__)
print("awkward version",ak.__version__)
print("numpy version",np.__version__)
print("matplotlib version",matplotlib.__version__)
print("hist version",hist.__version__)
print("vector version",vector.__version__)

uproot version 5.5.1
awkward version 2.7.2
numpy version 2.0.2
matplotlib version 3.9.0
hist version 2.8.0
vector version 1.5.2


In [2]:
vector.register_awkward()

In [3]:
DATATYPE="data"
assert((DATATYPE=="mc") or (DATATYPE=="data"))
BASEDIR="/pbs/throng/training/nantes-m2-rps-exp/data" # basedir where to look for runXXX.DATATYPE.root files
IS_MC=True if DATATYPE=="mc" else False

In [4]:
def data_file_path(run,is_mc=IS_MC,dest=BASEDIR):
    datatype="mc" if is_mc else "data"
    print({dest},"/run",{run},".",{datatype},".root")
    return f"{dest}/run{run}.{datatype}.root"

In [5]:
SAMPLE_RUNS=[291694,291399]

In [6]:
file = uproot.open("/pbs/throng/training/nantes-m2-rps-exp/data/run291263.mc.root")  # runXXX.mc.root pour les donnees simulees

gen = file["genTree"]
gen.show()

events = file["eventsTree"]

n = gen.arrays(["nMuonsGen","Muon_GenPx","Muon_GenPy","Muon_GenPz","Muon_GenE","Muon_GenLabel","Muon_GenMotherPDGCode"])

m = events.arrays(
    ["nMuons", "Muon_Px", "Muon_Py", "Muon_Pz", "Muon_Charge", "Muon_E","isCMUL","zVtx","Muon_matchedTrgThreshold","Muon_MCLabel"],
    how="zip")

name                 | typename                 | interpretation                
---------------------+--------------------------+-------------------------------
xVtxMC               | double                   | AsDtype('>f8')
yVtxMC               | double                   | AsDtype('>f8')
zVtxMC               | double                   | AsDtype('>f8')
nMuonsGen            | int32_t                  | AsDtype('>i4')
Muon_GenE            | std::vector<float>       | AsJagged(AsDtype('>f4'), he...
Muon_GenPx           | std::vector<float>       | AsJagged(AsDtype('>f4'), he...
Muon_GenPy           | std::vector<float>       | AsJagged(AsDtype('>f4'), he...
Muon_GenPz           | std::vector<float>       | AsJagged(AsDtype('>f4'), he...
Muon_GenLabel        | std::vector<int32_t>     | AsJagged(AsDtype('>i4'), he...
Muon_GenMotherPDG... | std::vector<int32_t>     | AsJagged(AsDtype('>i4'), he...


In [10]:
mask = (m["nMuons"] >= 2) & ak.all(n["Muon_GenMotherPDGCode"] == 443, axis=1) #permet de ne garder que le Jpsi et filtre les psi(2S)

291694


In [11]:
#boucle pour avoir le mask sur plusieurs run
nb_jpsi = []
for i in SAMPLE_RUNS :
    file = uproot.open(data_file_path(i,IS_MC))
    gen = file["genTree"]
    events = file["eventsTree"]
    n = gen.arrays(["nMuonsGen","Muon_GenPx","Muon_GenPy","Muon_GenPz","Muon_GenE","Muon_GenLabel","Muon_GenMotherPDGCode"])
    m = events.arrays(
    ["nMuons", "Muon_Px", "Muon_Py", "Muon_Pz", "Muon_Charge", "Muon_E","isCMUL","zVtx","Muon_matchedTrgThreshold","Muon_MCLabel"],
    how="zip")
    mask = (m["nMuons"] >= 2) & ak.all(n["Muon_GenMotherPDGCode"] == 443, axis=1) #permet de ne garder que le Jpsi et filtre les psi(2S)
    nb_jpsi.append(len(m[mask]))

print(nb_jpsi)

{'/pbs/throng/training/nantes-m2-rps-exp/data'} /run {291694} . {'data'} .root


KeyInFileError: not found: 'genTree' (with any cycle number)

    Available keys: 'eventsTree;1'

in file /pbs/throng/training/nantes-m2-rps-exp/data/run291694.data.root